In [1]:
import matplotlib.pyplot as plt
import numpy as np

In [2]:
sig = 10
ro = 28
beta = 8/3

x0 = np.array([1, 1, 1])

In [3]:
T1 = 0
T2 = 100
N = 10000
h = (abs(T1) + abs(T2))/N

t = [T1 + i*h for i in range(N+1)]
x = np.ndarray((N+1, 3))
x[0] = x0

In [4]:
def M(x):
    return np.array([
        sig * (x[1] - x[0]),
        x[0] * (ro - x[2]) - x[1],
        x[0]*x[1] - beta*x[2]
    ])

In [5]:
# Euler
def euler(x, h):
  return x + M(x) * h

In [6]:
# Runge-Kutta-2
def RK2(x, h):
    k1 = M(x)
    k2 = M(x + h * k1)
    return x + (h/2) * (k1 + k2)

def model_process(x, delta_t, h):
  while delta_t > h:
    x = RK4(x, h)
    delta_t -= h

  return x

In [7]:
# Runge-Kutta-4
def RK4(x, h):
  k1 = M(x)
  k2 = M(x + .5*h*k1)
  k3 = M(x + .5*h*k2)
  k4 = M(x + h*k3)

  return x + (h/6) * (k1 + 2*k2 + 2*k3 + k4)

In [8]:
def t2i(t):
  ''' Converts time into index'''
  return round(t/h)

In [9]:
# System process
for i in range(0, N):
    x[i+1] = RK4(x[i], h)

In [10]:
def J(x, xb, y0, R, B):
  '''Cost functional'''
  return 0.5*(np.dot(np.linalg.inv(B)@(x - xb), (x - xb)) + np.dot(np.linalg.inv(R)@(x - y0), (x - y0)))

In [11]:
def I(x_true, x_a):
  '''Squared distance between true vector and analysis vector'''
  return np.dot((x_true - x_a), (x_true - x_a))

In [12]:
np.random.seed(37)
R_t = np.eye(3)
B_t = np.eye(3)

TB = 45; T0 = 50

K = []
SC = []
x_b_initial = x[t2i(TB)]
x_b = model_process(x_b_initial, T0 - TB, h)
for i in range(1, 100):
  k = i/100
  temp = []
  for j in range(1000):
    x_true = x[t2i(T0)]
    y0 = np.random.multivariate_normal(x_true, R_t)


    x_b_spoiled = np.random.multivariate_normal(x_b, B_t)


    x_a = k*x_b_spoiled + (1-k)*y0


    B = np.eye(3) / k
    R = np.eye(3) / (1-k)
    temp.append(I(x_true, x_a))
  K.append(k)
  SC.append(sum(temp)/len(temp))

In [ ]:
plt.plot(K, SC)
plt.xlabel("κ")
plt.ylabel("Mean score")

'''As we see, minimum of difference between true vector and analysis vector is reached near κ = 0.5'''

In [ ]:
# Lorenz-63 attractor plot
fig = plt.figure(figsize=[7, 6])
ax = fig.add_subplot(projection='3d')

sc = ax.scatter(x[:,0], x[:,1], x[:,2], c=t, cmap='brg', s=0.1)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.colorbar(sc, label='Время', pad=0.1, ticks=[el for el in range(T1, T2+1, 10)])